# Deep Q Network (DQN)

Même principe que le QLearning.

L'estimation de la fonction Qualité se fait par un réseau au lieu de se faire par une table.

Il est donc tout à fait possible de construire une **table équivalente au réseau**, ce n'est pas la même structure de donnée mais elle représente la même fonction.


Lors de la mise à jour du réseau, de multiples cases de la table équivalente seront modifiées.

Ce phénomène est un **inconvénient** puisqu'en souhaitant apprendre quelque chose nous pouvons détruire ce qui a déjà été appris.

Mais c'est aussi un **avantage** puisqu'il est possible de généraliser notre apprentissage à d'autres situations est donc potentiellement accélérer l'apprentissage.

DQN partage certains paramètres avec le QLearning :
- **exploration_initial_eps** : identique à eps_start
- **exploration_final_eps** : identique à eps_end
- **exploration_fraction** : identique à eps_fraction
- **gamma** : identique à gamma

Attention !

Le paramètre **learning_rate** n'est pas identique au alpha du QLearning (même s'il a le même nom)

L'utilisation d'un réseau introduit de nouveaux paramètres :
- **learning_rate** : le pas effectué à chaque rétropropagation du gradient
- **buffer_size** : la taille du buffer
- **batch_size** : la taille du batch
- **train_freq** : le nombre de steps entre deux mise à jour du réseau online
- **gradient_steps** : le nombre de fois que l'on crée un batch puis rétropropage une fois la moyenne des erreurs de ses éléments dans le réseau online.
- **target_update_interval** : l'interval entre deux mise à jour du réseau target
- **net_arch** : l'architecture du réseau (le nombre de couches intermédiaires et le nombre de noeuds par couche)


net_arch = [64, 32] signifie que l'on a 2 couches intermédiaires, la première comporte 64 noeuds et la seconde en a 32.

**Par défaut** :
- **learning_rate** : 0.0001
- **buffer_size** : 1000000
- **batch_size** : 32
- **gamma** : 0.99
- **train_freq** : 4
- **gradient_steps** : 1
- **target_update_interval** : 10000
- **exploration_fraction** : 0.1
- **exploration_initial_eps** : 1.0
- **exploration_final_eps** : 0.05
- **net_arch** : [64, 64]

# DQN de StableBaselines : Entraînement

## Utilisation basique

On commence par créer une instance de l'algorithme que l'on utilise pour l'entraînement.

Lors de la création de l'instance de DQN, le premier paramètre est dans la majorité des cas "**MlpPolicy**".
Si l'**observation_space** est de type **spaces.Dict** alors il faut utiliser "**MultiInputPolicy**".

In [ ]:
from stable_baselines3 import DQN

env = instance de mon environnement

algorithme = DQN(
    "MlpPolicy",  # utiliser "MultiInputPolicy" si l'observation_space est spaces.Dict
    env
)

model = algorithme.learn(total_timesteps = 500)

## Obtenir le gain par épisode : le logger

Si on souhaite obtenir le gain par épisode pendant l'entraînement, il est nécessaire de créer un logger.

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class RewardLogger(BaseCallback) :

    # Le constructeur du logger
    def __init__(self) :
        super().__init__()
        self.rewards = []  # Pour le gain par épisode
        self.current_reward = 0 # Pour le gain de l'épisode en cours


    # Appelé à chaque step d'entraînement
    def _on_step(self) :
        self.current_reward += self.locals["rewards"][0]  # Ajoute la dernière récompense obtenue
        if self.locals["dones"][0] :  # Vérifie si l'épisode est terminé
            self.rewards.append(self.current_reward)
            self.current_reward = 0
        return True

# L'instance du logger que nous utiliserons.
logger = RewardLogger()

Une fois le logger créé, nous pouvons en faire usage.

In [ ]:
from stable_baselines3 import DQN
from notre_fichier_python import RewardLogger


env = instance de mon environnement

algorithme = DQN(
    "MlpPolicy",  # utiliser "MultiInputPolicy" si l'observation_space est spaces.Dict
    env
)



logger = RewardLogger()


model = algorithme.learn(
    total_timesteps = 500,
    callback = logger
)


print(f"Gain par épiode : {logger.rewards}")

## Paramétrer DQN

Voyons comment spécifier les paramètres de DQN

In [ ]:
from stable_baselines3 import DQN

env = instance de mon environnement



policy_kwargs = dict(net_arch = [8, 4])  # On souhaite 2 couches intermédiaires de 8 puis 4 noeuds


algorithme = DQN(
    "MlpPolicy",  # utiliser "MultiInputPolicy" si l'observation_space est spaces.Dict
    env,
    policy_kwargs = policy_kwargs,
    learning_rate = 0.001,
    buffer_size = 100000,
    target_update_interval = 500
)



model = algorithme.learn(total_timesteps = 500)

# DQN de StableBaselines : Exploitation

Voici comment utiliser un modèle obtenu après entraînement

In [ ]:
from stable_baselines3.common.env_util import make_vec_env

vec_env = make_vec_env(lambda : env, n_envs = 1)  # n_envs = 1 pour ne pas faire de parallélisme

obs = vec_env.reset()
done = False
gain = 0

# Commenter la ligne ci-dessous pour ne pas effectuer l'affichage
vec_env.env_method('render') #Comme on travail avec le vec_env, on ne peut pas faire vec_env.render()


# Ajouter un compteur pour éviter les boucles infinies si nécessaire
while not done :
    action, _ = model.predict(obs, deterministic = True) # Attention au deterministic = True !!

    obs, reward, done, info = vec_env.step(action)
    gain += reward

    # Commenter la ligne ci-dessous pour ne pas effectuer l'affichage
    vec_env.env_method('render')

print(f"Gain obtenu : {gain}")

Attention, dès que l'épisode est terminé, le vec_env se reset immédiatement, ce programme ne permet donc pas d'afficher l'état final ayant terminé l'épisode.

# Quels environnements sont compatibles avec DQN ?

Tous les environnements ayant un action_space de type : spaces.Discrete(n)